In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

test = pd.read_csv('../data/test_final.csv')
X_test = test.drop(columns=['isFraud'])
y_test = test['isFraud']

best_model = xgb.XGBClassifier()
best_model.load_model('../models/xgb_tuned.json')

with open('../models/best_threshold.json') as f:
    threshold = json.load(f)['threshold']

y_prob = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= threshold).astype(int)

print(f"Test size: {X_test.shape}")
print(f"Fraud predicted: {y_pred.sum()}")

Test size: (2000, 192)
Fraud predicted: 39


In [2]:
print("Tính SHAP values... (1-2 phút)")

explainer = shap.TreeExplainer(best_model)

# Dùng 500 mẫu để nhanh hơn
sample_idx = np.random.choice(len(X_test), 500, replace=False)
X_sample = X_test.iloc[sample_idx]

shap_values = explainer.shap_values(X_sample)

print(f"SHAP values shape: {shap_values.shape}")
print("✅ Tính xong!")

Tính SHAP values... (1-2 phút)
SHAP values shape: (500, 192)
✅ Tính xong!


In [3]:
# Top 20 features quan trọng nhất
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, 
    X_sample,
    max_display=20,
    show=False
)
plt.title('SHAP Summary Plot — Top 20 Features')
plt.tight_layout()
plt.savefig('../reports/shap_summary.png', dpi=80, bbox_inches='tight')
plt.close()
print("✅ Đã lưu shap_summary.png!")

✅ Đã lưu shap_summary.png!


In [4]:
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values,
    X_sample,
    plot_type='bar',
    max_display=15,
    show=False
)
plt.title('SHAP Feature Importance — Top 15')
plt.tight_layout()
plt.savefig('../reports/shap_bar.png', dpi=80, bbox_inches='tight')
plt.close()
print("✅ Đã lưu shap_bar.png!")

✅ Đã lưu shap_bar.png!


In [5]:
# Tìm 1 giao dịch fraud thật trong sample
fraud_indices = np.where(y_test.iloc[sample_idx].values == 1)[0]

if len(fraud_indices) > 0:
    fraud_idx = fraud_indices[0]
    
    explanation = shap.Explanation(
        values=shap_values[fraud_idx],
        base_values=explainer.expected_value,
        data=X_sample.iloc[fraud_idx],
        feature_names=X_test.columns.tolist()
    )
    
    plt.figure(figsize=(10, 6))
    shap.waterfall_plot(explanation, max_display=15, show=False)
    plt.title('SHAP Waterfall — Giải thích 1 giao dịch FRAUD')
    plt.tight_layout()
    plt.savefig('../reports/shap_waterfall.png', dpi=80, bbox_inches='tight')
    plt.close()
    print(f"✅ Đã lưu shap_waterfall.png!")
    print(f"   Giao dịch này có xác suất fraud: {y_prob[sample_idx[fraud_idx]]:.4f}")
else:
    print("Không tìm thấy fraud trong sample, tăng sample size lên!")

✅ Đã lưu shap_waterfall.png!
   Giao dịch này có xác suất fraud: 0.0002


In [7]:
# Tính mean absolute SHAP
mean_shap = np.abs(shap_values).mean(axis=0)
feature_importance = pd.DataFrame({
    'feature': X_test.columns,
    'importance': mean_shap
}).sort_values('importance', ascending=False).head(10)

print("🔍 TOP 10 FEATURES QUAN TRỌNG NHẤT:")
print("="*45)
for i, row in feature_importance.iterrows():
    print(f"  {row['feature']:<25} {row['importance']:.4f}")

print("\n💡 Ý nghĩa:")
print("  - Số càng cao = feature càng ảnh hưởng đến kết quả")
print("  - Màu đỏ trong summary plot = giá trị cao → tăng risk fraud")
print("  - Màu xanh = giá trị thấp → giảm risk fraud")

🔍 TOP 10 FEATURES QUAN TRỌNG NHẤT:
  card6                     1.1631
  TransactionAmt            0.9448
  C4                        0.8003
  M4                        0.6184
  card1_freq                0.6143
  card1                     0.5913
  C1                        0.5766
  D10                       0.4580
  C2                        0.4491
  M6                        0.4230

💡 Ý nghĩa:
  - Số càng cao = feature càng ảnh hưởng đến kết quả
  - Màu đỏ trong summary plot = giá trị cao → tăng risk fraud
  - Màu xanh = giá trị thấp → giảm risk fraud
